# 11 · Lista de casa: uma demonstração para imagens

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Você vai preparar a demonstração da rede convolucional da aula 08, que classifica peças de roupa. O usuário envia uma foto qualquer; a rede só conhece imagens de 28 × 28 pixels, em tons de cinza, com fundo preto. Fazer a ponte entre as duas coisas, e ser honesto sobre o que acontece quando ela falha, é o trabalho desta lista.

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- A primeira célula baixa os dados e treina a rede por três épocas (menos de um minuto em CPU).
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Preparar a imagem que o usuário envia | 2,5 |
| 2 | Função de previsão e interface | 2,0 |
| 3 | O que acontece fora do laboratório | 2,5 |
| 4 | "Sobre este modelo" | 2,0 |
| 5 | Publicação | 1,0 |

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)

import gzip
import warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn

SEMENTE = 42
torch.manual_seed(SEMENTE)

BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/"
PASTA = Path("fashion_mnist")
PASTA.mkdir(exist_ok=True)
for arquivo in ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]:
    if not (PASTA / arquivo).exists():
        urlretrieve(BASE + arquivo, PASTA / arquivo)


def ler_imagens(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)


def ler_rotulos(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)


imagens, rotulos = ler_imagens(PASTA / "train-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "train-labels-idx1-ubyte.gz")
imagens_teste, rotulos_teste = ler_imagens(PASTA / "t10k-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "t10k-labels-idx1-ubyte.gz")
NOMES = ["Camiseta", "Calça", "Suéter", "Vestido", "Casaco", "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]
MEDIA, DESVIO = float(imagens.mean() / 255), float(imagens.std() / 255)


def padronizar(lote_uint8):
    """Array uint8 (N, 28, 28), fundo preto -> tensor float32 (N, 1, 28, 28) padronizado, como no treino."""
    return ((torch.from_numpy(np.array(lote_uint8)).float() / 255 - MEDIA) / DESVIO).unsqueeze(1)


modelo = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10),
)
otimizador, funcao_perda = torch.optim.Adam(modelo.parameters(), lr=1e-3), nn.CrossEntropyLoss()
X_treino, y_treino = padronizar(imagens), torch.from_numpy(rotulos.copy()).long()
for epoca in range(3):
    for indices in torch.randperm(len(X_treino)).split(128):
        otimizador.zero_grad()
        funcao_perda(modelo(X_treino[indices]), y_treino[indices]).backward()
        otimizador.step()
modelo.eval()


@torch.no_grad()
def prever_lote(lote_uint8):
    """Classe prevista para cada imagem de um array uint8 (N, 28, 28)."""
    return modelo(padronizar(lote_uint8)).argmax(dim=1).numpy()


acuracia_teste = float((prever_lote(imagens_teste) == rotulos_teste).mean())
print(f"gradio {gr.__version__} · rede treinada · acurácia no teste: {acuracia_teste:.1%}")

## Questão 1 (2,5 pontos): preparar a imagem que o usuário envia

O componente `gr.Image` entrega à sua função um array NumPy `uint8`, com forma `(altura, largura, 3)` para imagens coloridas ou `(altura, largura)` para imagens em tons de cinza, de qualquer tamanho.

Escreva `preparar_imagem(imagem)`, que devolve um tensor `float32` de forma `(1, 1, 28, 28)`, pronto para a rede:

1. converta o array em imagem do Pillow (`Image.fromarray`) e passe para tons de cinza (`.convert("L")`);
2. redimensione para 28 × 28 (`.resize((28, 28))`);
3. volte para array, divida por 255 e padronize com `MEDIA` e `DESVIO`;
4. acrescente os eixos de lote e de canal.

In [ ]:
# ✏️ Questão 1
def preparar_imagem(imagem):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
_original = imagens_teste[0]                                               # uma bota, 28 × 28, tons de cinza
_grande_colorida = np.repeat(np.kron(_original, np.ones((10, 10), dtype=np.uint8))[:, :, None], 3, axis=2)      # a mesma, 280 × 280 × 3

_a = preparar_imagem(_original)
assert tuple(_a.shape) == (1, 1, 28, 28) and _a.dtype == torch.float32, "confira a forma e o tipo do tensor"
assert torch.allclose(_a, padronizar(_original[None]), atol=1e-5), "para uma imagem que já é 28 × 28 em tons de cinza, o resultado deve ser igual ao do treino"
_b = preparar_imagem(_grande_colorida)
assert tuple(_b.shape) == (1, 1, 28, 28) and (_b - _a).abs().mean() < 0.05, "a versão ampliada e colorida deveria resultar em quase a mesma coisa"
assert _original.max() == 255 and imagens_teste[0] is not None and _original.dtype == np.uint8, "a função não deve alterar a imagem recebida"

fig, eixos = plt.subplots(1, 2, figsize=(4.5, 2.3))
eixos[0].imshow(_grande_colorida); eixos[0].set_title("recebida: 280 × 280 × 3", fontsize=9)
eixos[1].imshow(_b[0, 0], cmap="gray"); eixos[1].set_title("entregue à rede: 28 × 28", fontsize=9)
for ax in eixos:
    ax.axis("off")
print("Formato correto.")

**1b)** Por que a padronização precisa usar **os mesmos** valores de `MEDIA` e `DESVIO` do treino, e não a média e o desvio da imagem enviada? O que mais, além da padronização, precisa ser igual entre o treino e o uso para que a rede funcione?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): função de previsão e interface

a) Escreva `classificar(imagem)`: se `imagem` for `None`, levanta `gr.Error` pedindo uma imagem; caso contrário, devolve um dicionário com a probabilidade de **cada uma das 10 classes** (`{nome da classe: probabilidade}`, com valores `float` do Python), usando `preparar_imagem`, a rede e `torch.softmax`.

b) Monte `demo_q2`: uma `gr.Interface` com entrada `gr.Image(type="numpy")` e saída `gr.Label(num_top_classes=3)`, com título e descrição.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
def classificar(imagem):
    ...


demo_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
_saida = classificar(imagens_teste[0])
assert list(_saida) == NOMES and all(type(v) is float for v in _saida.values()), "o dicionário deve ter as 10 classes, com valores float"
assert abs(sum(_saida.values()) - 1) < 1e-4, "as probabilidades devem somar 1"
_diretas = prever_lote(imagens_teste[:100])
_pela_funcao = [int(np.argmax(list(classificar(img).values()))) for img in imagens_teste[:100]]
assert (np.array(_pela_funcao) == _diretas).all(), "a classe mais provável deve coincidir com a previsão direta da rede"
try:
    classificar(None)
    raise AssertionError("sem imagem, a função deveria levantar gr.Error")
except gr.Error:
    pass
assert isinstance(demo_q2, gr.Interface)
print("Formato correto. Para abrir a interface: demo_q2.launch()")

## Questão 3 (2,5 pontos): o que acontece fora do laboratório

A rede acerta quase 90% no conjunto de teste. As imagens que um usuário envia não são do conjunto de teste. Vamos simular quatro diferenças comuns e medir.

Escreva as quatro transformações abaixo. Cada uma recebe um array `uint8` de forma `(N, 28, 28)` e devolve outro com a mesma forma e o mesmo tipo.

- `inverter`: fundo claro e objeto escuro (255 menos a imagem), como em uma foto de catálogo;
- `deslocar`: todas as imagens deslocadas 5 pixels para a direita, com a faixa que sobra à esquerda preenchida com zero;
- `girar`: todas as imagens giradas 90 graus (`np.rot90`, com `axes=(1, 2)`);
- `ruido`: soma de ruído normal com desvio padrão 40 (use `np.random.default_rng(0)`), com o resultado limitado ao intervalo de 0 a 255.

Depois monte `tabela_q3`: uma `Series` com a acurácia em `imagens_teste` para `"original"`, `"invertida"`, `"deslocada"`, `"girada"` e `"com ruído"`, nessa ordem. Use `prever_lote`.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
def inverter(lote):
    ...

def deslocar(lote, pixels=5):
    ...

def girar(lote):
    ...

def ruido(lote):
    ...

tabela_q3 = ...

In [ ]:
# ✅ Conferência da questão 3 (não altere)
_amostra = imagens_teste[:20]
for _f in (inverter, deslocar, girar, ruido):
    _r = _f(_amostra)
    assert _r.shape == _amostra.shape and _r.dtype == np.uint8, f"{_f.__name__} deve devolver uint8 com a mesma forma"
assert (inverter(_amostra).astype(int) + _amostra == 255).all()
assert (deslocar(_amostra)[:, :, :5] == 0).all() and (deslocar(_amostra)[:, :, 5:] == _amostra[:, :, :-5]).all(), "confira o deslocamento para a direita"
assert girar(_amostra)[0].sum() == _amostra[0].sum() and not (girar(_amostra)[0] == _amostra[0]).all()
assert 10 < np.abs(ruido(_amostra).astype(int) - _amostra).mean() < 36 and (_amostra == imagens_teste[:20]).all(), "confira o ruído (e não altere as imagens originais)"
assert list(tabela_q3.index) == ["original", "invertida", "deslocada", "girada", "com ruído"] and tabela_q3.between(0, 1).all()
assert abs(tabela_q3["original"] - acuracia_teste) < 1e-9

fig, eixos = plt.subplots(1, 5, figsize=(9, 2.2))
for ax, (nome, figura) in zip(eixos, [("original", _amostra), ("invertida", inverter(_amostra)), ("deslocada", deslocar(_amostra)), ("girada", girar(_amostra)), ("com ruído", ruido(_amostra))]):
    ax.imshow(figura[0], cmap="gray", vmin=0, vmax=255); ax.axis("off"); ax.set_title(f"{nome}\n{tabela_q3[nome]:.0%}", fontsize=9)
print("Formato correto. Acurácia no teste em cada condição:")
print(tabela_q3.round(3).to_string())

**3b)** Qual transformação mais prejudica a rede? Para uma pessoa, alguma dessas imagens ficou difícil de reconhecer? Explique a diferença: por que a rede falha em algo que, para nós, é a mesma peça de roupa?

**3c)** A foto que um usuário tira com o celular se parece mais com qual dessas condições? Proponha duas medidas para lidar com isso: uma no **preparo da imagem** (na aplicação) e uma no **treino** do modelo.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): "Sobre este modelo"

a) Escreva em `SOBRE_Q4` (uma string em Markdown, com no máximo 150 palavras) o texto que apareceria na tela da demonstração. Ele deve dizer: com que dados a rede foi treinada; quão bem ela funciona, e em que condições esse número vale; em que condições ela falha, **usando números da questão 3**; e para que a demonstração não deve ser usada.

b) Descreva uma entrada que a aplicação deveria **recusar ou sinalizar** em vez de classificar, e como você detectaria esse caso no código.

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
SOBRE_Q4 = """
...
"""

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert isinstance(SOBRE_Q4, str) and 40 <= len(SOBRE_Q4.split()) <= 150, "o texto deve ter entre 40 e 150 palavras"
assert any(c.isdigit() for c in SOBRE_Q4), "inclua números da questão 3"
print("Formato correto. Texto como apareceria na tela:\n")
print(SOBRE_Q4)

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): publicação

a) Liste os arquivos que você colocaria na pasta da aplicação para publicá-la, e o que cada um contém. Nesta rede, o artefato seria um `state_dict` gravado com `torch.save`: o que mais, além dele, a aplicação precisa ter para reconstruir o modelo? Compare com o `.joblib` da aula.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._